In [1]:
import cv2
import numpy as np
import time
from math import atan2, degrees

video_path = "people.mp4"

trackers = [('CSRT', cv2.TrackerCSRT_create), 
            ('KCF', cv2.TrackerKCF_create), 
            ('MOSSE', cv2.legacy.TrackerMOSSE_create)]

all_results = []

for tracker_name, tracker_func in trackers:
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    
    print(f"\n=== {tracker_name} ===")
    bbox = cv2.selectROI(f"{tracker_name} - выберите объект", frame, False)
    cv2.destroyAllWindows()
    
    if bbox != (0,0,0,0):
        tracker = tracker_func()
        tracker.init(frame, bbox)
        
        traj = []
        x,y,w,h = bbox
        prev = (x+w//2, y+h//2)
        
        frames = 0
        lost_frames = 0
        speeds = []
        start = time.time()
        
        while True:
            ret, frame = cap.read()
            if not ret: break
            
            ok, b = tracker.update(frame)
            
            if ok:
                x,y,w,h = [int(v) for v in b]
                curr = (x+w//2, y+h//2)
                traj.append(curr)
                if len(traj) > 30: traj.pop(0)
                
                dx, dy = curr[0]-prev[0], curr[1]-prev[1]
                sp = (dx*dx+dy*dy)**0.5
                speeds.append(sp)
                ang = degrees(atan2(dy,dx)) if sp>0 else 0
                prev = curr
                
                cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
                cv2.circle(frame, curr, 4, (0,0,255), -1)
                if len(traj) > 1:
                    cv2.polylines(frame, [np.array(traj, dtype=np.int32)], False, (0,255,255), 2)
                cv2.putText(frame, f"{tracker_name} | Speed:{sp:.1f} | Angle:{ang:.0f}", (10,30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
            else:
                lost_frames += 1
                cv2.putText(frame, "LOST", (300,240), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
            
            cv2.imshow(tracker_name, cv2.resize(frame, (640,480)))
            frames += 1
            if cv2.waitKey(1) & 0xFF == 27: break
        
        cap.release()
        cv2.destroyAllWindows()
        
        fps = frames / (time.time()-start)
        all_results.append([tracker_name, round(fps,1), round(sum(speeds)/len(speeds),1) if speeds else 0, lost_frames])
        print(f"{tracker_name}: FPS={fps:.1f}, Потеряно={lost_frames}")

print("\n=== РЕЗУЛЬТАТЫ ===")
print("Трекер   | FPS | Скорость | Потеряно")
print("-" * 40)
for r in all_results:
    print(f"{r[0]:8} | {r[1]:4} | {r[2]:7} | {r[3]}")


=== CSRT ===
CSRT: FPS=19.2, Потеряно=0

=== KCF ===
KCF: FPS=26.8, Потеряно=37

=== MOSSE ===
MOSSE: FPS=64.3, Потеряно=0

=== РЕЗУЛЬТАТЫ ===
Трекер   | FPS | Скорость | Потеряно
----------------------------------------
CSRT     | 19.2 |     1.1 | 0
KCF      | 26.8 |     1.9 | 37
MOSSE    | 64.3 |     0.8 | 0


In [1]:
import cv2
import numpy as np

video_path = "people.mp4"

cap = cv2.VideoCapture(video_path)
ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
prev_gray = cv2.GaussianBlur(prev_gray, (21, 21), 0)

tracker = None
tracking = False

print("Детектор движения запущен. Ожидание движущегося объекта...")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)
    
    if not tracking:
        diff = cv2.absdiff(prev_gray, gray)
        thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)[1]
        thresh = cv2.dilate(thresh, None, iterations=2)
        
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        for contour in contours:
            area = cv2.contourArea(contour)
            if area > 500:
                x, y, w, h = cv2.boundingRect(contour)
                bbox = (x, y, w, h)
                tracker = cv2.TrackerCSRT_create()
                tracker.init(frame, bbox)
                tracking = True
                print(f"Объект обнаружен! Начинаем трекинг: {bbox}")
                break
    
    if tracking:
        success, bbox = tracker.update(frame)
        if success:
            x, y, w, h = [int(v) for v in bbox]
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(frame, "AUTO TRACKING", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        else:
            tracking = False
            print("Объект потерян, снова ждём движение...")
    
    cv2.imshow("Motion Detector + Auto Tracking", cv2.resize(frame, (640, 480)))
    
    prev_gray = gray
    
    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
print("Завершено")

Детектор движения запущен. Ожидание движущегося объекта...
Объект обнаружен! Начинаем трекинг: (356, 454, 62, 20)
Завершено


In [2]:
import cv2
import numpy as np

video_path = "people.mp4"

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()

trackers = []
colors = [(0, 255, 0), (255, 0, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]

num_objects = int(input("Сколько объектов отслеживать? "))

for i in range(num_objects):
    print(f"Выберите объект {i+1}")
    bbox = cv2.selectROI(f"Объект {i+1}", frame, False)
    cv2.destroyWindow(f"Объект {i+1}")
    
    if bbox != (0, 0, 0, 0):
        tracker = cv2.TrackerCSRT_create()
        tracker.init(frame, bbox)
        trackers.append({
            'tracker': tracker,
            'bbox': bbox,
            'color': colors[i % len(colors)],
            'trajectory': []
        })

print(f"Отслеживается объектов: {len(trackers)}")
print("Нажмите ESC для выхода")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    active_trackers = []
    
    for t in trackers:
        success, bbox = t['tracker'].update(frame)
        
        if success:
            active_trackers.append(t)
            x, y, w, h = [int(v) for v in bbox]
            center = (x + w//2, y + h//2)
            
            t['trajectory'].append(center)
            if len(t['trajectory']) > 30:
                t['trajectory'].pop(0)
            
            cv2.rectangle(frame, (x, y), (x + w, y + h), t['color'], 2)
            cv2.circle(frame, center, 4, t['color'], -1)
            
            if len(t['trajectory']) > 1:
                points = np.array(t['trajectory'], dtype=np.int32)
                cv2.polylines(frame, [points], False, t['color'], 2)
    
    trackers = active_trackers
    
    cv2.putText(frame, f"Objects: {len(trackers)}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.imshow("Multi-Object Tracking", cv2.resize(frame, (640, 480)))
    
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
print("Многообъектный трекинг завершен")

Сколько объектов отслеживать?  2


Выберите объект 1
Выберите объект 2
Отслеживается объектов: 2
Нажмите ESC для выхода
Многообъектный трекинг завершен
